# 01 - Data audit (Dataset A: Slither Audited Smart Contracts)
Blueprint P1.2. Answers: how many contracts, did Slither succeed, what does label 4 mean, which detectors occur, and does a random sample of contracts look right?

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option('display.max_colwidth', 120)
T = ROOT / 'reports' / 'tables'
F = ROOT / 'reports' / 'figures'

## Filtering funnel

In [2]:
pd.read_csv(T / 'dataset_funnel.csv')

,step,contracts
0,rows_in_raw_shards,120608
1,after_dropping_duplicate_addresses,120608
2,after_dropping_failed_slither_runs,120608
3,after_dropping_empty_or_tiny_sources,120510
4,after_exact_deduplication,65058


## The label-4 question
We re-create the integer ids the upstream loading script emits for each configuration from the raw Slither output. In the *big* and *small* configurations id 4 is `safe` and never co-occurs with another id. In the *all* configuration id 4 is `divide-before-multiply`, it does co-occur with other ids, and ids go up to 38 (`safe`). The dataset viewer shows the *all* configuration, which explains the confusing rows noted in the blueprint.

In [3]:
pd.read_csv(T / 'label4_audit.csv')

,config,contracts,label_names,id4_name,id4_share,id4_with_other_ids_share,max_id_seen,safe_id
0,big,106474,6,safe,0.339473,0.000000,5,4
1,small,14134,9,safe,0.212254,0.000000,8,4
2,all,120608,39,divide-before-multiply,0.166191,0.137114,38,38


## Slither detectors and label families (de-duplicated pool)

In [4]:
pd.read_csv(T / 'detector_frequency.csv').head(40)

,detector,contracts,share,family
0,unused-return,25456,0.391282,unchecked-calls
1,reentrancy-no-eth,14237,0.218836,reentrancy
2,uninitialized-local,13532,0.207999,other
3,divide-before-multiply,12453,0.191414,arithmetic
4,incorrect-equality,9967,0.153202,other
5,unchecked-transfer,9397,0.144440,unchecked-calls
6,reentrancy-eth,6867,0.105552,reentrancy
7,arbitrary-send,6197,0.095254,access-control
8,locked-ether,3287,0.050524,locked-ether
9,erc20-interface,2970,0.045652,ignore


In [5]:
pd.read_csv(T / 'family_distribution.csv')

,family,positives,prevalence,kept_as_class
0,unchecked-calls,31144,0.478711,True
1,other,22446,0.345015,True
2,reentrancy,18710,0.287590,True
3,safe,17900,0.275139,False
4,arithmetic,12453,0.191414,True
5,access-control,9654,0.148391,True
6,locked-ether,3287,0.050524,True
7,bad-randomness,2503,0.038473,True
8,double-spending,368,0.005656,False


## Ten random contracts with their labels (sanity check)

In [6]:
df = pd.read_parquet(ROOT / 'data/processed/dataset.parquet', columns=['address','contract_name','labels','detectors','n_lines','snippet'])
sample = df.sample(10, random_state=42)
for r in sample.itertuples():
    print(f'{r.address}  {r.contract_name}  lines={r.n_lines}')
    print('  detectors:', list(r.detectors)); print('  classes  :', list(r.labels))
    print('  ' + '\n  '.join(r.snippet.splitlines()[:4])); print()

0xfDcdfA378818AC358739621ddFa8582E6ac1aDcB  Csc  lines=150
  detectors: []
  classes  : []
  contract Csc is LibNote {
      mapping (address => uint) public wards;
      function rely(address guy) external note auth { wards[guy] = 1; }
      function deny(address guy) external note auth { wards[guy] = 0; }

0x4E6D068F43Ae28ce4A7BC97D9dc5FDfB8a55f008  WitnetRequestBoardTrustableDefault  lines=2834
  detectors: ['uninitialized-local', 'unused-return', 'controlled-delegatecall', 'arbitrary-send']
  classes  : ['access-control', 'other', 'unchecked-calls']
  contract WitnetRequestBoardTrustableDefault
      is
          Destructible,
          WitnetRequestBoardTrustableBase

0xc846330b2e34481d374203aef5c980d2762d4893  CityLoot  lines=1659
  detectors: ['divide-before-multiply', 'reentrancy-no-eth', 'incorrect-shift', 'write-after-write', 'weak-prng', 'unused-return']
  classes  : ['arithmetic', 'bad-randomness', 'reentrancy', 'unchecked-calls']
  contract CityLoot is ERC721Enumerable, Re